# nb24 — 頑健性チューニング実験

## 評価基準（CVスコアは参考値のみ）

| 指標 | 説明 | 良い方向 |
|------|------|---------|
| seed_std | 5seedのtest予測の標準偏差（平均） | 小さい |
| overfit_gap | val_RMSE_le170 - train_RMSE_le170 | 小さい |
| fold_std | Fold1,2,4,5のRMSE_le170のStd | 小さい |
| test_mean | testの予測平均（健全範囲 40-55%） | 40-55% |
| gt170 | test予測が>170の件数 | 0に近い |
| CV_mean | RMSE_le170の fold平均（参考） | — |

## 判定基準
**seed_std↓ AND overfit_gap↓ AND 分布健全** を同時に満たす場合のみ「改善候補」。
CVが良くなっても上記を満たさなければ不採用。

## 実験軸
- **Axis 1 CNN**: Dropout 0.3→0.5 / weight_decay 0→1e-3 / patience 20→10
- **Axis 2 MLP**: Dropout 0.4→0.5 / weight_decay 1e-4→1e-3 / patience 20→10
- **Axis 3 ET**: max_depth None→15 / min_samples_leaf 1→5 / max_features 0.3→0.2+n_est→500

In [1]:
import sys, os, copy, time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T = 200.0

# CV seeds (same as baseline nb23)
CV_SEEDS = [42, 123, 456]
# Stability seeds: 5 different seeds to measure seed-to-seed variance
STABILITY_SEEDS = [42, 123, 456, 789, 999]

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta,  _,   X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

def snv_sg1(R):
    A = R.copy().astype(np.float64)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)

X_pp    = snv_sg1(X_raw)
X_pp_te = snv_sg1(X_test_raw)
N_IN    = X_pp.shape[1]

def rmse_le(yt, yp, T=170.0):
    yt, yp = np.asarray(yt), np.asarray(yp)
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan

def rmse_all(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt)-np.asarray(yp))**2)))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'CV_SEEDS: {CV_SEEDS}  STABILITY_SEEDS: {STABILITY_SEEDS}')
print(f'N_IN={N_IN}')

Device: cpu
Train: (1322, 1555)  Test: (550, 1555)
CV_SEEDS: [42, 123, 456]  STABILITY_SEEDS: [42, 123, 456, 789, 999]
N_IN=1555


In [2]:
# ===== Parameterized models =====

class ImprovedCNN1D(nn.Module):
    """Dropout rate as parameter for regularization tuning."""
    def __init__(self, dropout=0.3):
        super().__init__()
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.conv3     = nn.Sequential(nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)
        self.pool      = nn.AdaptiveAvgPool1d(16)
        self.fc = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(32*16, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.block12(x.unsqueeze(1))
        h = torch.relu(self.conv3(h) + self.shortcut3(h))
        return self.fc(self.pool(h).view(x.size(0), -1)).squeeze(1)

class ShallowMLP(nn.Module):
    """Dropout rate as parameter for regularization tuning."""
    def __init__(self, n_in=N_IN, hidden=128, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 32),   nn.BatchNorm1d(32),    nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

n_cnn = sum(p.numel() for p in ImprovedCNN1D().parameters())
n_mlp = sum(p.numel() for p in ShallowMLP(N_IN).parameters())
print(f'ImprovedCNN1D params: {n_cnn:,}')
print(f'ShallowMLP    params: {n_mlp:,}')

ImprovedCNN1D params: 20,993
ShallowMLP    params: 203,649


In [3]:
# ===== Training functions =====

def train_nn_cv(ModelClass, model_kw, Xtr_s, ytr, Xva_s, yva, seed,
                lr=1e-3, wd=0.0, patience=20, n_epochs=100, batch=32, use_huber=False):
    """
    CV training with overfitting gap measurement.
    Returns (val_pred, tr_rmse_le, va_rmse_le, stop_ep)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    ytr32 = ytr.astype(np.float32)
    yva32 = yva.astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr32).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva32).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model  = ModelClass(**model_kw).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit   = nn.HuberLoss(delta=10.0) if use_huber else nn.MSELoss()

    best_val, best_state = float('inf'), None
    no_improve = 0; stop_ep = n_epochs
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xva_t), yva_t).item()
        if vl < best_val:
            best_val = vl; best_state = copy.deepcopy(model.state_dict()); no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            stop_ep = epoch + 1; break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(Xva_t).cpu().numpy()
        tr_pred  = model(Xtr_t).cpu().numpy()

    return val_pred, rmse_le(ytr, tr_pred), rmse_le(yva, val_pred), stop_ep


def train_nn_full_fixed(ModelClass, model_kw, Xtr_s, ytr, seed,
                        n_epochs=70, lr=1e-3, wd=0.0, batch=32, use_huber=False):
    """Full-train for seed stability (fixed epochs, no early stopping)."""
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t  = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t  = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model  = ModelClass(**model_kw).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    crit   = nn.HuberLoss(delta=10.0) if use_huber else nn.MSELoss()
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    model.eval()
    return model


def train_et_cv(Xtr, ytr, Xva, yva, **et_kw):
    """ET CV with overfitting gap. Returns (val_pred, tr_rmse_le, va_rmse_le)."""
    et = ExtraTreesRegressor(**et_kw)
    et.fit(Xtr, ytr)
    tr_pred  = et.predict(Xtr)
    val_pred = et.predict(Xva)
    return val_pred, rmse_le(ytr, tr_pred), rmse_le(yva, val_pred)


print('Training functions defined.')

Training functions defined.


In [4]:
# ===== Evaluation helpers =====

def nn_cv_eval(label, ModelClass, model_kw, nn_kw):
    """Run CV for a NN config. Returns metrics dict including OOF array."""
    print(f'  [{label}] CV...', flush=True)
    fold_va_rmse = []; fold_tr_rmse = []; stops = []
    oof_preds = np.zeros(len(y))

    for fi, (tr, va) in enumerate(SPLITS):
        Xtr_pp, Xva_pp = X_pp[tr], X_pp[va]
        ytr, yva       = y[tr], y[va]
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
        Xva_s = sc.transform(Xva_pp).astype(np.float32)

        seed_val_preds, seed_tr, seed_stops = [], [], []
        for seed in CV_SEEDS:
            vp, tr_r, va_r, stop = train_nn_cv(ModelClass, model_kw, Xtr_s, ytr,
                                                Xva_s, yva, seed, **nn_kw)
            seed_val_preds.append(vp); seed_tr.append(tr_r); seed_stops.append(stop)

        avg_vp = np.mean(seed_val_preds, 0)
        oof_preds[va] = avg_vp
        fold_va_rmse.append(rmse_le(yva, avg_vp))
        fold_tr_rmse.append(float(np.mean(seed_tr)))
        stops.append(float(np.mean(seed_stops)))
        print(f'    F{fi+1}: va={fold_va_rmse[-1]:.2f}%  tr={fold_tr_rmse[-1]:.2f}%  stop={stops[-1]:.0f}', flush=True)

    no3 = [0,1,3,4]
    fold_std  = float(np.std([fold_va_rmse[i] for i in no3]))
    cv_mean   = float(np.mean(fold_va_rmse))
    avg_gap   = float(np.mean([fold_va_rmse[i]-fold_tr_rmse[i] for i in no3]))
    avg_stop  = int(round(np.mean(stops)))
    return dict(fold_va=fold_va_rmse, fold_tr=fold_tr_rmse, fold_std=fold_std,
                cv_mean=cv_mean, overfit_gap=avg_gap, avg_stop=avg_stop, oof=oof_preds)


def nn_stability_eval(label, ModelClass, model_kw, nn_kw, avg_stop):
    """Seed stability on full train. Returns (seed_std, test_mean, gt170)."""
    print(f'  [{label}] Stability ({avg_stop} ep)...', flush=True)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(X_pp).astype(np.float32)
    Xte_s = sc.transform(X_pp_te).astype(np.float32)
    Xte_t = torch.from_numpy(Xte_s).to(DEVICE)

    te_preds = []
    for seed in STABILITY_SEEDS:
        model = train_nn_full_fixed(ModelClass, model_kw, Xtr_s, y, seed,
                                    n_epochs=avg_stop, **{k:v for k,v in nn_kw.items()
                                                          if k in ('lr','wd','batch','use_huber')})
        with torch.no_grad():
            te_preds.append(model(Xte_t).cpu().numpy())
    te_arr   = np.array(te_preds)          # [5, 550]
    seed_std = float(te_arr.std(axis=0).mean())
    te_mean  = float(np.clip(te_arr.mean(0), 0, CLIP_T).mean())
    gt170    = int((np.clip(te_arr.mean(0), 0, CLIP_T) > 170).sum())
    return seed_std, te_mean, gt170


def et_cv_eval(label, et_kw_base):
    """Run CV for an ET config. Returns metrics dict including OOF array."""
    print(f'  [{label}] ET CV...', flush=True)
    fold_va_rmse = []; fold_tr_rmse = []
    oof_preds = np.zeros(len(y))
    for fi, (tr, va) in enumerate(SPLITS):
        vp, tr_r, va_r = train_et_cv(X_pp[tr], y[tr], X_pp[va], y[va], **et_kw_base)
        oof_preds[va] = vp
        fold_va_rmse.append(va_r); fold_tr_rmse.append(tr_r)
        print(f'    F{fi+1}: va={va_r:.2f}%  tr={tr_r:.2f}%', flush=True)
    no3 = [0,1,3,4]
    fold_std = float(np.std([fold_va_rmse[i] for i in no3]))
    cv_mean  = float(np.mean(fold_va_rmse))
    avg_gap  = float(np.mean([fold_va_rmse[i]-fold_tr_rmse[i] for i in no3]))
    return dict(fold_va=fold_va_rmse, fold_tr=fold_tr_rmse, fold_std=fold_std,
                cv_mean=cv_mean, overfit_gap=avg_gap, oof=oof_preds)


def et_stability_eval(label, et_kw_base):
    """Seed stability for ET (random_state varies)."""
    print(f'  [{label}] ET Stability...', flush=True)
    te_preds = []
    for seed in STABILITY_SEEDS:
        kw = dict(et_kw_base); kw['random_state'] = seed
        et = ExtraTreesRegressor(**kw)
        et.fit(X_pp, y)
        te_preds.append(et.predict(X_pp_te))
    te_arr   = np.array(te_preds)
    seed_std = float(te_arr.std(axis=0).mean())
    te_mean  = float(np.clip(te_arr.mean(0), 0, CLIP_T).mean())
    gt170    = int((np.clip(te_arr.mean(0), 0, CLIP_T) > 170).sum())
    return seed_std, te_mean, gt170


def print_results_table(rows):
    """rows: list of dict with keys: label, seed_std, overfit_gap, fold_std, test_mean, gt170, cv_mean"""
    hdr = f'  {"Config":<28} | {"seed_std":>8} | {"gap":>7} | {"fld_std":>7} | {"te_mean":>7} | {">170":>4} | {"CV_mean":>7}'
    print(hdr)
    print('  ' + '-'*83)
    for r in rows:
        flag = '← BASE' if r.get('is_base') else ''
        healthy = 40 <= r['test_mean'] <= 55 and r['gt170'] <= 5
        dist_ok = '' if healthy else ' ⚠dist'
        print(f'  {r["label"]:<28} | {r["seed_std"]:>8.3f} | {r["overfit_gap"]:>7.2f} '
              f'| {r["fold_std"]:>7.2f} | {r["test_mean"]:>7.1f} | {r["gt170"]:>4} '
              f'| {r["cv_mean"]:>7.2f}  {flag}{dist_ok}')


print('Helpers defined. Starting experiments...')
print(f'Estimated runtime: CNN x4={4*20} fits, MLP x4={4*20} fits, ET x4={4*10} fits')

Helpers defined. Starting experiments...
Estimated runtime: CNN x4=80 fits, MLP x4=80 fits, ET x4=40 fits


## 仮確定版 Baseline (nb23設定)

In [5]:
print('=' * 70)
print('BASELINE (仮確定版 — nb23設定)')
print('CNN: Dropout=0.3, wd=0, patience=20')
print('MLP: Dropout=0.4, wd=1e-4, patience=20')
print('ET : n_est=300, mf=0.3, depth=None, msl=1')
print('=' * 70)
t0 = time.time()

# --- CNN baseline ---
cnn_base_mk = dict(dropout=0.3)
cnn_base_nk = dict(lr=1e-3, wd=0.0, patience=20, n_epochs=100, use_huber=True)
base_cnn_cv  = nn_cv_eval('CNN base', ImprovedCNN1D, cnn_base_mk, cnn_base_nk)
base_cnn_ss  = nn_stability_eval('CNN base', ImprovedCNN1D, cnn_base_mk, cnn_base_nk, base_cnn_cv['avg_stop'])

# --- MLP baseline ---
mlp_base_mk = dict(n_in=N_IN, hidden=128, dropout=0.4)
mlp_base_nk = dict(lr=1e-3, wd=1e-4, patience=20, n_epochs=100, use_huber=False)
base_mlp_cv  = nn_cv_eval('MLP base', ShallowMLP, mlp_base_mk, mlp_base_nk)
base_mlp_ss  = nn_stability_eval('MLP base', ShallowMLP, mlp_base_mk, mlp_base_nk, base_mlp_cv['avg_stop'])

# --- ET baseline ---
et_base_kw = dict(n_estimators=300, max_features=0.3, max_depth=None,
                  min_samples_leaf=1, random_state=42, n_jobs=-1)
base_et_cv  = et_cv_eval('ET base', et_base_kw)
base_et_ss  = et_stability_eval('ET base', et_base_kw)

# Use OOF already computed above (no double run)
oof_3avg_base = np.clip((base_et_cv['oof'] + base_cnn_cv['oof'] + base_mlp_cv['oof']) / 3, 0, CLIP_T)
base_ens_rmse = [rmse_le(y[va], oof_3avg_base[va]) for _, va in SPLITS]
base_ens_std  = float(np.std([base_ens_rmse[i] for i in [0,1,3,4]]))
base_ens_mean = float(np.mean(base_ens_rmse))

print(f'\nBaseline done in {(time.time()-t0)/60:.1f}min')
print()
print('-- 仮確定版ベースライン結果 --')
print(f'CNN: seed_std={base_cnn_ss[0]:.3f}  gap={base_cnn_cv["overfit_gap"]:.2f}  fld_std={base_cnn_cv["fold_std"]:.2f}  te_mean={base_cnn_ss[1]:.1f}  >170={base_cnn_ss[2]}  CV={base_cnn_cv["cv_mean"]:.2f}%')
print(f'MLP: seed_std={base_mlp_ss[0]:.3f}  gap={base_mlp_cv["overfit_gap"]:.2f}  fld_std={base_mlp_cv["fold_std"]:.2f}  te_mean={base_mlp_ss[1]:.1f}  >170={base_mlp_ss[2]}  CV={base_mlp_cv["cv_mean"]:.2f}%')
print(f'ET : seed_std={base_et_ss[0]:.3f}  gap={base_et_cv["overfit_gap"]:.2f}  fld_std={base_et_cv["fold_std"]:.2f}  te_mean={base_et_ss[1]:.1f}  >170={base_et_ss[2]}  CV={base_et_cv["cv_mean"]:.2f}%')
print(f'3種アンサンブル: Std(F1245)={base_ens_std:.2f}%  mean={base_ens_mean:.2f}%')

BASELINE (仮確定版 — nb23設定)
CNN: Dropout=0.3, wd=0, patience=20
MLP: Dropout=0.4, wd=1e-4, patience=20
ET : n_est=300, mf=0.3, depth=None, msl=1
  [CNN base] CV...
    F1: va=13.12%  tr=10.51%  stop=56
    F2: va=20.65%  tr=14.92%  stop=35
    F3: va=46.71%  tr=46.88%  stop=21
    F4: va=18.98%  tr=14.26%  stop=36
    F5: va=10.35%  tr=7.71%  stop=62
  [CNN base] Stability (42 ep)...
  [MLP base] CV...
    F1: va=15.34%  tr=7.78%  stop=74
    F2: va=19.22%  tr=7.92%  stop=58
    F3: va=43.83%  tr=7.06%  stop=69
    F4: va=19.84%  tr=16.25%  stop=56
    F5: va=11.41%  tr=6.39%  stop=89
  [MLP base] Stability (69 ep)...
  [ET base] ET CV...
    F1: va=18.95%  tr=0.00%
    F2: va=13.75%  tr=0.00%
    F3: va=25.72%  tr=0.00%
    F4: va=12.32%  tr=0.00%
    F5: va=10.19%  tr=0.00%
  [ET base] ET Stability...

Baseline done in 18.7min

-- 仮確定版ベースライン結果 --
CNN: seed_std=2.544  gap=3.93  fld_std=4.20  te_mean=46.2  >170=5  CV=21.96%
MLP: seed_std=1.848  gap=6.87  fld_std=3.38  te_mean=46.3  >170=0

## Axis 1: CNN 正則化

In [6]:
print('=' * 70)
print('AXIS 1: CNN 正則化')
print('=' * 70)

cnn_configs = [
    ('CNN p=0.3 wd=0 pt=20 (BASE)', dict(dropout=0.3), dict(lr=1e-3, wd=0.0, patience=20, n_epochs=100, use_huber=True), True),
    ('CNN p=0.5 wd=0 pt=20',        dict(dropout=0.5), dict(lr=1e-3, wd=0.0, patience=20, n_epochs=100, use_huber=True), False),
    ('CNN p=0.3 wd=1e-3 pt=20',     dict(dropout=0.3), dict(lr=1e-3, wd=1e-3, patience=20, n_epochs=100, use_huber=True), False),
    ('CNN p=0.3 wd=0 pt=10',        dict(dropout=0.3), dict(lr=1e-3, wd=0.0, patience=10,  n_epochs=100, use_huber=True), False),
]

cnn_rows = []
cnn_cv_results = {}
t1 = time.time()

for label, mk, nk, is_base in cnn_configs:
    if is_base:
        cv_r = base_cnn_cv; ss = base_cnn_ss
    else:
        cv_r = nn_cv_eval(label, ImprovedCNN1D, mk, nk)
        ss   = nn_stability_eval(label, ImprovedCNN1D, mk, nk, cv_r['avg_stop'])
    cnn_cv_results[label] = (cv_r, ss, mk, nk)
    cnn_rows.append(dict(label=label, seed_std=ss[0], overfit_gap=cv_r['overfit_gap'],
                         fold_std=cv_r['fold_std'], test_mean=ss[1], gt170=ss[2],
                         cv_mean=cv_r['cv_mean'], is_base=is_base))
    print(f'  [{label}] stop={cv_r["avg_stop"]}ep  folds={[round(x,1) for x in cv_r["fold_va"]]}')

print(f'\nAxis 1 done in {(time.time()-t1)/60:.1f}min')
print()
print('=== Axis 1: CNN 正則化 結果 ===')
print_results_table(cnn_rows)

# 所見
base_cnn_row = cnn_rows[0]
print()
print('所見:')
for r in cnn_rows[1:]:
    ss_better   = r['seed_std']    < base_cnn_row['seed_std']
    gap_better  = r['overfit_gap'] < base_cnn_row['overfit_gap']
    dist_ok     = 40 <= r['test_mean'] <= 55 and r['gt170'] <= 5
    if ss_better and gap_better and dist_ok:
        verdict = '改善候補 ← seed安定性↑ & gap↓ & 分布健全'
    elif ss_better or gap_better:
        verdict = 'トレードオフ — 改悪リスクあり'
    else:
        verdict = '改善なし'
    print(f'  {r["label"]}: {verdict}')

AXIS 1: CNN 正則化
  [CNN p=0.3 wd=0 pt=20 (BASE)] stop=42ep  folds=[13.1, 20.7, 46.7, 19.0, 10.4]
  [CNN p=0.5 wd=0 pt=20] CV...
    F1: va=12.65%  tr=14.76%  stop=36
    F2: va=20.50%  tr=15.55%  stop=36
    F3: va=47.79%  tr=48.00%  stop=21
    F4: va=19.52%  tr=16.04%  stop=27
    F5: va=10.86%  tr=8.02%  stop=79
  [CNN p=0.5 wd=0 pt=20] Stability (40 ep)...
  [CNN p=0.5 wd=0 pt=20] stop=40ep  folds=[12.6, 20.5, 47.8, 19.5, 10.9]
  [CNN p=0.3 wd=1e-3 pt=20] CV...
    F1: va=13.04%  tr=10.57%  stop=51
    F2: va=21.03%  tr=14.96%  stop=35
    F3: va=46.90%  tr=46.94%  stop=21
    F4: va=18.98%  tr=14.41%  stop=36
    F5: va=10.31%  tr=7.71%  stop=62
  [CNN p=0.3 wd=1e-3 pt=20] Stability (41 ep)...
  [CNN p=0.3 wd=1e-3 pt=20] stop=41ep  folds=[13.0, 21.0, 46.9, 19.0, 10.3]
  [CNN p=0.3 wd=0 pt=10] CV...
    F1: va=13.48%  tr=11.62%  stop=35
    F2: va=22.40%  tr=17.84%  stop=14
    F3: va=46.71%  tr=46.88%  stop=11
    F4: va=20.33%  tr=17.27%  stop=15
    F5: va=10.53%  tr=7.83%  stop=

## Axis 2: MLP 正則化

In [7]:
print('=' * 70)
print('AXIS 2: MLP 正則化')
print('=' * 70)

mlp_configs = [
    ('MLP p=0.4 wd=1e-4 pt=20 (BASE)', dict(n_in=N_IN, hidden=128, dropout=0.4), dict(lr=1e-3, wd=1e-4, patience=20, n_epochs=100, use_huber=False), True),
    ('MLP p=0.5 wd=1e-4 pt=20',        dict(n_in=N_IN, hidden=128, dropout=0.5), dict(lr=1e-3, wd=1e-4, patience=20, n_epochs=100, use_huber=False), False),
    ('MLP p=0.4 wd=1e-3 pt=20',        dict(n_in=N_IN, hidden=128, dropout=0.4), dict(lr=1e-3, wd=1e-3, patience=20, n_epochs=100, use_huber=False), False),
    ('MLP p=0.4 wd=1e-4 pt=10',        dict(n_in=N_IN, hidden=128, dropout=0.4), dict(lr=1e-3, wd=1e-4, patience=10,  n_epochs=100, use_huber=False), False),
]

mlp_rows = []
mlp_cv_results = {}
t2 = time.time()

for label, mk, nk, is_base in mlp_configs:
    if is_base:
        cv_r = base_mlp_cv; ss = base_mlp_ss
    else:
        cv_r = nn_cv_eval(label, ShallowMLP, mk, nk)
        ss   = nn_stability_eval(label, ShallowMLP, mk, nk, cv_r['avg_stop'])
    mlp_cv_results[label] = (cv_r, ss, mk, nk)
    mlp_rows.append(dict(label=label, seed_std=ss[0], overfit_gap=cv_r['overfit_gap'],
                         fold_std=cv_r['fold_std'], test_mean=ss[1], gt170=ss[2],
                         cv_mean=cv_r['cv_mean'], is_base=is_base))
    print(f'  [{label}] stop={cv_r["avg_stop"]}ep  folds={[round(x,1) for x in cv_r["fold_va"]]}')

print(f'\nAxis 2 done in {(time.time()-t2)/60:.1f}min')
print()
print('=== Axis 2: MLP 正則化 結果 ===')
print_results_table(mlp_rows)

base_mlp_row = mlp_rows[0]
print()
print('所見:')
for r in mlp_rows[1:]:
    ss_better   = r['seed_std']    < base_mlp_row['seed_std']
    gap_better  = r['overfit_gap'] < base_mlp_row['overfit_gap']
    dist_ok     = 40 <= r['test_mean'] <= 55 and r['gt170'] <= 5
    if ss_better and gap_better and dist_ok:
        verdict = '改善候補 ← seed安定性↑ & gap↓ & 分布健全'
    elif ss_better or gap_better:
        verdict = 'トレードオフ — 改悪リスクあり'
    else:
        verdict = '改善なし'
    print(f'  {r["label"]}: {verdict}')

AXIS 2: MLP 正則化
  [MLP p=0.4 wd=1e-4 pt=20 (BASE)] stop=69ep  folds=[15.3, 19.2, 43.8, 19.8, 11.4]
  [MLP p=0.5 wd=1e-4 pt=20] CV...
    F1: va=14.84%  tr=6.95%  stop=86
    F2: va=18.21%  tr=9.47%  stop=58
    F3: va=40.99%  tr=10.59%  stop=54
    F4: va=16.72%  tr=9.29%  stop=68
    F5: va=11.33%  tr=6.52%  stop=86
  [MLP p=0.5 wd=1e-4 pt=20] Stability (70 ep)...
  [MLP p=0.5 wd=1e-4 pt=20] stop=70ep  folds=[14.8, 18.2, 41.0, 16.7, 11.3]
  [MLP p=0.4 wd=1e-3 pt=20] CV...
    F1: va=15.56%  tr=8.26%  stop=64
    F2: va=18.79%  tr=10.26%  stop=53
    F3: va=43.45%  tr=6.13%  stop=71
    F4: va=19.40%  tr=11.06%  stop=64
    F5: va=11.49%  tr=6.28%  stop=82
  [MLP p=0.4 wd=1e-3 pt=20] Stability (67 ep)...
  [MLP p=0.4 wd=1e-3 pt=20] stop=67ep  folds=[15.6, 18.8, 43.4, 19.4, 11.5]
  [MLP p=0.4 wd=1e-4 pt=10] CV...
    F1: va=15.43%  tr=9.28%  stop=54
    F2: va=19.22%  tr=7.92%  stop=48
    F3: va=43.20%  tr=16.00%  stop=39
    F4: va=20.27%  tr=16.85%  stop=42
    F5: va=11.47%  tr=6.57

## Axis 3: ET 過適合抑制

In [8]:
print('=' * 70)
print('AXIS 3: ET 過適合抑制')
print('=' * 70)

et_configs = [
    ('ET n300 mf0.3 d=None msl=1 (BASE)', dict(n_estimators=300, max_features=0.3, max_depth=None, min_samples_leaf=1, random_state=42, n_jobs=-1), True),
    ('ET n300 mf0.3 d=15 msl=1',          dict(n_estimators=300, max_features=0.3, max_depth=15,   min_samples_leaf=1, random_state=42, n_jobs=-1), False),
    ('ET n300 mf0.3 d=None msl=5',        dict(n_estimators=300, max_features=0.3, max_depth=None, min_samples_leaf=5, random_state=42, n_jobs=-1), False),
    ('ET n500 mf0.2 d=None msl=1',        dict(n_estimators=500, max_features=0.2, max_depth=None, min_samples_leaf=1, random_state=42, n_jobs=-1), False),
]

et_rows = []
et_cv_results = {}
t3 = time.time()

for label, kw, is_base in et_configs:
    if is_base:
        cv_r = base_et_cv; ss = base_et_ss
    else:
        cv_r = et_cv_eval(label, kw)
        ss   = et_stability_eval(label, kw)
    et_cv_results[label] = (cv_r, ss, kw)
    et_rows.append(dict(label=label, seed_std=ss[0], overfit_gap=cv_r['overfit_gap'],
                        fold_std=cv_r['fold_std'], test_mean=ss[1], gt170=ss[2],
                        cv_mean=cv_r['cv_mean'], is_base=is_base))
    print(f'  [{label}] folds={[round(x,1) for x in cv_r["fold_va"]]}')

print(f'\nAxis 3 done in {(time.time()-t3)/60:.1f}min')
print()
print('=== Axis 3: ET 過適合抑制 結果 ===')
print_results_table(et_rows)

base_et_row = et_rows[0]
print()
print('所見:')
for r in et_rows[1:]:
    ss_better   = r['seed_std']    < base_et_row['seed_std']
    gap_better  = r['overfit_gap'] < base_et_row['overfit_gap']
    dist_ok     = 40 <= r['test_mean'] <= 55 and r['gt170'] <= 5
    if ss_better and gap_better and dist_ok:
        verdict = '改善候補 ← seed安定性↑ & gap↓ & 分布健全'
    elif ss_better or gap_better:
        verdict = 'トレードオフ — 改悪リスクあり'
    else:
        verdict = '改善なし'
    print(f'  {r["label"]}: {verdict}')

AXIS 3: ET 過適合抑制
  [ET n300 mf0.3 d=None msl=1 (BASE)] folds=[18.9, 13.7, 25.7, 12.3, 10.2]
  [ET n300 mf0.3 d=15 msl=1] ET CV...
    F1: va=19.03%  tr=0.02%
    F2: va=13.60%  tr=0.05%
    F3: va=25.94%  tr=0.02%
    F4: va=12.24%  tr=0.08%
    F5: va=10.46%  tr=0.04%
  [ET n300 mf0.3 d=15 msl=1] ET Stability...
  [ET n300 mf0.3 d=15 msl=1] folds=[19.0, 13.6, 25.9, 12.2, 10.5]
  [ET n300 mf0.3 d=None msl=5] ET CV...
    F1: va=18.59%  tr=0.96%
    F2: va=13.74%  tr=1.05%
    F3: va=25.45%  tr=1.02%
    F4: va=12.60%  tr=0.97%
    F5: va=10.03%  tr=0.99%
  [ET n300 mf0.3 d=None msl=5] ET Stability...
  [ET n300 mf0.3 d=None msl=5] folds=[18.6, 13.7, 25.5, 12.6, 10.0]
  [ET n500 mf0.2 d=None msl=1] ET CV...
    F1: va=18.56%  tr=0.00%
    F2: va=12.82%  tr=0.00%
    F3: va=24.91%  tr=0.00%
    F4: va=13.15%  tr=0.00%
    F5: va=9.88%  tr=0.00%
  [ET n500 mf0.2 d=None msl=1] ET Stability...
  [ET n500 mf0.2 d=None msl=1] folds=[18.6, 12.8, 24.9, 13.1, 9.9]

Axis 3 done in 3.4min

=== Axi

## 総合サマリー & 改善候補選定

In [9]:
print('=' * 70)
print('総合サマリー')
print('=' * 70)
print()

# ---- 改善候補の自動選定 ----
def best_config(rows, base_row):
    """seed_std↓ AND gap↓ AND dist_ok を同時に満たす最良設定を返す。"""
    candidates = []
    for r in rows:
        if r.get('is_base'):
            continue
        ss_better  = r['seed_std']    < base_row['seed_std']
        gap_better = r['overfit_gap'] < base_row['overfit_gap']
        dist_ok    = 40 <= r['test_mean'] <= 55 and r['gt170'] <= 5
        if ss_better and gap_better and dist_ok:
            candidates.append(r)
    if not candidates:
        return None
    # 最良: seed_stdの改善量最大
    return min(candidates, key=lambda x: x['seed_std'])

best_cnn = best_config(cnn_rows, cnn_rows[0])
best_mlp = best_config(mlp_rows, mlp_rows[0])
best_et  = best_config(et_rows,  et_rows[0])

print('改善候補:')
print(f'  CNN: {best_cnn["label"] if best_cnn else "なし → 仮確定維持 (Dropout=0.3, wd=0, patience=20)"}')
print(f'  MLP: {best_mlp["label"] if best_mlp else "なし → 仮確定維持 (Dropout=0.4, wd=1e-4, patience=20)"}')
print(f'  ET : {best_et["label"]  if best_et  else "なし → 仮確定維持 (n_est=300, mf=0.3, depth=None, msl=1)"}')
print()

any_improved = best_cnn or best_mlp or best_et
if not any_improved:
    print('結論: いずれの軸でも頑健性が明確に向上しなかった。')
    print('      仮確定版 (ET+CNN+MLP 等重み) を最終提出として維持する。')
else:
    print('結論: 改善候補あり。以下でチューニング版アンサンブルを生成する。')

print()
print('--- 全指標の一覧 ---')
print()
print('【CNN軸】')
print_results_table(cnn_rows)
print()
print('【MLP軸】')
print_results_table(mlp_rows)
print()
print('【ET軸】')
print_results_table(et_rows)

総合サマリー

改善候補:
  CNN: なし → 仮確定維持 (Dropout=0.3, wd=0, patience=20)
  MLP: なし → 仮確定維持 (Dropout=0.4, wd=1e-4, patience=20)
  ET : ET n500 mf0.2 d=None msl=1

結論: 改善候補あり。以下でチューニング版アンサンブルを生成する。

--- 全指標の一覧 ---

【CNN軸】
  Config                       | seed_std |     gap | fld_std | te_mean | >170 | CV_mean
  -----------------------------------------------------------------------------------
  CNN p=0.3 wd=0 pt=20 (BASE)  |    2.544 |    3.93 |    4.20 |    46.2 |    5 |   21.96  ← BASE
  CNN p=0.5 wd=0 pt=20         |    2.953 |    2.29 |    4.19 |    44.4 |    1 |   22.26  
  CNN p=0.3 wd=1e-3 pt=20      |    3.401 |    3.93 |    4.34 |    46.2 |    5 |   22.05  
  CNN p=0.3 wd=0 pt=10         |    4.682 |    3.05 |    4.85 |    48.2 |    7 |   22.69   ⚠dist

【MLP軸】
  Config                       | seed_std |     gap | fld_std | te_mean | >170 | CV_mean
  -----------------------------------------------------------------------------------
  MLP p=0.4 wd=1e-4 pt=20 (BASE) |    1.848 |    6.87 

## チューニング版アンサンブル生成 (改善候補がある場合)

In [10]:
import os
os.makedirs('../submissions', exist_ok=True)

# ---- 選択した設定でアンサンブルを構築 ----
# 改善候補がない軸はベースライン設定を使用

# CNNの選択設定
if best_cnn:
    sel_cnn_label = best_cnn['label']
    _, _, sel_cnn_mk, sel_cnn_nk = cnn_cv_results[sel_cnn_label]
else:
    sel_cnn_label = '仮確定'
    sel_cnn_mk, sel_cnn_nk = cnn_base_mk, cnn_base_nk

# MLPの選択設定
if best_mlp:
    sel_mlp_label = best_mlp['label']
    _, _, sel_mlp_mk, sel_mlp_nk = mlp_cv_results[sel_mlp_label]
else:
    sel_mlp_label = '仮確定'
    sel_mlp_mk, sel_mlp_nk = mlp_base_mk, mlp_base_nk

# ETの選択設定
if best_et:
    sel_et_label = best_et['label']
    _, _, sel_et_kw = et_cv_results[sel_et_label]
    # update random_state=42 (stability eval uses varied seeds)
    sel_et_kw = dict(sel_et_kw); sel_et_kw['random_state'] = 42
else:
    sel_et_label = '仮確定'
    sel_et_kw = et_base_kw

print(f'チューニング版アンサンブル構成:')
print(f'  CNN: {sel_cnn_label}')
print(f'  MLP: {sel_mlp_label}')
print(f'  ET : {sel_et_label}')
print()

if not any_improved:
    print('改善候補がないためチューニング版CSVは生成しない。')
    print('仮確定版 sub_et_cnn_mlp_avg.csv を最終提出候補として維持。')
else:
    print('チューニング版 CV計算中...')
    oof_t_cnn = np.zeros(len(y))
    oof_t_mlp = np.zeros(len(y))
    oof_t_et  = np.zeros(len(y))

    for fi, (tr, va) in enumerate(SPLITS):
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(X_pp[tr]).astype(np.float32)
        Xva_s = sc.transform(X_pp[va]).astype(np.float32)

        sp = [train_nn_cv(ImprovedCNN1D, sel_cnn_mk, Xtr_s, y[tr], Xva_s, y[va], s,
                          **sel_cnn_nk)[0] for s in CV_SEEDS]
        oof_t_cnn[va] = np.mean(sp, 0)

        sp = [train_nn_cv(ShallowMLP, sel_mlp_mk, Xtr_s, y[tr], Xva_s, y[va], s,
                          **sel_mlp_nk)[0] for s in CV_SEEDS]
        oof_t_mlp[va] = np.mean(sp, 0)

        oof_t_et[va], _, _ = train_et_cv(X_pp[tr], y[tr], X_pp[va], y[va], **sel_et_kw)
        print(f'  Fold {fi+1} done', flush=True)

    oof_t_3avg = np.clip((oof_t_et + oof_t_cnn + oof_t_mlp) / 3, 0, CLIP_T)
    tuned_folds = [rmse_le(y[va], oof_t_3avg[va]) for _, va in SPLITS]
    tuned_std   = float(np.std([tuned_folds[i] for i in [0,1,3,4]]))
    tuned_mean  = float(np.mean(tuned_folds))

    print(f'\nチューニング版 CV:')
    print(f'  Folds: {[round(x,2) for x in tuned_folds]}')
    print(f'  Mean={tuned_mean:.2f}%  Std(F1245)={tuned_std:.2f}%')
    print(f'  比較: 仮確定版 Std={base_ens_std:.2f}%  Mean={base_ens_mean:.2f}%')
    print()

    # Test predictions (avg_stop from selected config CV)
    sel_cnn_stop = cnn_cv_results[sel_cnn_label][0]['avg_stop'] if best_cnn else base_cnn_cv['avg_stop']
    sel_mlp_stop = mlp_cv_results[sel_mlp_label][0]['avg_stop'] if best_mlp else base_mlp_cv['avg_stop']

    sc_full = StandardScaler()
    Xtr_s_f = sc_full.fit_transform(X_pp).astype(np.float32)
    Xte_s_f = sc_full.transform(X_pp_te).astype(np.float32)
    Xte_tf  = torch.from_numpy(Xte_s_f).to(DEVICE)

    cnn_te_sp = []
    for seed in CV_SEEDS:
        m = train_nn_full_fixed(ImprovedCNN1D, sel_cnn_mk, Xtr_s_f, y, seed,
                                n_epochs=sel_cnn_stop, wd=sel_cnn_nk['wd'],
                                use_huber=sel_cnn_nk['use_huber'])
        with torch.no_grad():
            cnn_te_sp.append(m(Xte_tf).cpu().numpy())
    te_cnn_t = np.clip(np.mean(cnn_te_sp, 0), 0, CLIP_T)

    mlp_te_sp = []
    for seed in CV_SEEDS:
        m = train_nn_full_fixed(ShallowMLP, sel_mlp_mk, Xtr_s_f, y, seed,
                                n_epochs=sel_mlp_stop, wd=sel_mlp_nk['wd'],
                                use_huber=False)
        with torch.no_grad():
            mlp_te_sp.append(m(Xte_tf).cpu().numpy())
    te_mlp_t = np.clip(np.mean(mlp_te_sp, 0), 0, CLIP_T)

    et_full = ExtraTreesRegressor(**sel_et_kw)
    et_full.fit(X_pp, y)
    te_et_t = np.clip(et_full.predict(X_pp_te), 0, CLIP_T)

    te_tuned = np.clip((te_et_t + te_cnn_t + te_mlp_t) / 3, 0, CLIP_T)
    print(f'Test dist (tuned): mean={te_tuned.mean():.1f}%  >170={(te_tuned>170).sum()}')

    make_submission(test_meta, te_tuned, '../submissions/sub_robust_tuned.csv')
    print('Saved: sub_robust_tuned.csv')

チューニング版アンサンブル構成:
  CNN: 仮確定
  MLP: 仮確定
  ET : ET n500 mf0.2 d=None msl=1

チューニング版 CV計算中...
  Fold 1 done
  Fold 2 done
  Fold 3 done
  Fold 4 done
  Fold 5 done

チューニング版 CV:
  Folds: [13.82, 16.01, 32.76, 16.06, 9.94]
  Mean=17.72%  Std(F1245)=2.49%
  比較: 仮確定版 Std=2.45%  Mean=17.79%

Test dist (tuned): mean=45.2%  >170=0
Saved submission: ../submissions/sub_robust_tuned.csv
Saved: sub_robust_tuned.csv


In [11]:
print()
print('=' * 70)
print('最終サマリー & 提出推奨')
print('=' * 70)
print()
print('判定基準: seed_std↓ AND overfit_gap↓ AND 分布健全 を同時に満たすか')
print()

if not any_improved:
    print('結果: いずれの軸でも3指標同時改善なし。')
    print('      → 仮確定版を最終提出として維持する。')
    print()
    print('提出候補 (最終2枠):')
    print('  ① sub_et_cnn_mlp_avg.csv  (仮確定 ET+CNN+MLP等重み, OOF Std=2.44%)')
    print('  ② ET単体 sub_et_sns_sg41.csv (LB=18.35 — 独立保険)')
    print()
    print('注: Public=17-18点台に収まるか要確認。CVスコアでなく頑健性指標で判断。')
else:
    print('結果: 改善候補あり → sub_robust_tuned.csv を追加提出候補とする。')
    print()
    if tuned_std < base_ens_std:
        print(f'  チューニング版 Std={tuned_std:.2f}% < 仮確定版 Std={base_ens_std:.2f}% → 頑健性向上')
    else:
        print(f'  チューニング版 Std={tuned_std:.2f}% vs 仮確定版 Std={base_ens_std:.2f}% — Std改善なし（要注意）')
    print()
    print('提出候補 (最終2枠):')
    print('  ① sub_robust_tuned.csv  (頑健チューニング版 — Public 17-18点台に収まるか確認)')
    print('  ② sub_et_cnn_mlp_avg.csv  または ET単体 (独立保険)')
    print()
    print('注: CVが多少悪化しても、seed安定性↑&分布健全なら頑健性向上とみなす。')
    print('    Public提出で崩れないか必ず確認すること。')

print()
print(f'実験完了。総経過時間: {(time.time()-t0)/60:.1f}min')


最終サマリー & 提出推奨

判定基準: seed_std↓ AND overfit_gap↓ AND 分布健全 を同時に満たすか

結果: 改善候補あり → sub_robust_tuned.csv を追加提出候補とする。

  チューニング版 Std=2.49% vs 仮確定版 Std=2.45% — Std改善なし（要注意）

提出候補 (最終2枠):
  ① sub_robust_tuned.csv  (頑健チューニング版 — Public 17-18点台に収まるか確認)
  ② sub_et_cnn_mlp_avg.csv  または ET単体 (独立保険)

注: CVが多少悪化しても、seed安定性↑&分布健全なら頑健性向上とみなす。
    Public提出で崩れないか必ず確認すること。

実験完了。総経過時間: 73.5min
